# 00 — Preparação do Dataset

Este notebook organiza as imagens brutas em splits `train/val/test` estratificados por classe
e realiza análise exploratória (EDA) do dataset de doenças de soja.

**Classes:** `ferrugem-asiatica`, `mancha-alvo`, `antracnose`, `cercosporiose`, `mildio`, `saudavel`

## Estrutura esperada após organização
```
data/
  train/{classe}/*.jpg
  val/{classe}/*.jpg
  test/{classe}/*.jpg
```

## 1. Setup

In [ ]:
# Instala dependências extras no Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q matplotlib seaborn Pillow tqdm
    # Clona o repositório
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from PIL import Image
from tqdm import tqdm

CLASSES = [
    'ferrugem-asiatica',
    'mancha-alvo',
    'antracnose',
    'cercosporiose',
    'mildio',
    'saudavel',
]
print('Classes:', CLASSES)

## 2. Montar Google Drive e definir caminhos

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

# ============================================================
# CONFIGURE AQUI:
# RAW_DIR: pasta com imagens brutas organizadas por subpasta de classe
# DATA_DIR: onde os splits train/val/test serão criados
# ============================================================
RAW_DIR  = Path('/content/drive/MyDrive/ze-praga-dataset/raw')
DATA_DIR = Path('/content/drive/MyDrive/ze-praga-dataset')

print(f'Raw:  {RAW_DIR}')
print(f'Data: {DATA_DIR}')

## 3. Verificar imagens brutas

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

raw_counts = {}
for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    if cls_dir.exists():
        imgs = [p for p in cls_dir.iterdir() if p.suffix in EXTENSIONS]
        raw_counts[cls] = len(imgs)
    else:
        raw_counts[cls] = 0
        print(f'  AVISO: pasta não encontrada: {cls_dir}')

total = sum(raw_counts.values())
print(f'\nTotal de imagens brutas: {total}')
for cls, count in raw_counts.items():
    print(f'  {cls:<22}: {count:>5} imagens')

## 4. Verificar integridade (imagens corrompidas)

In [ ]:
corrupted = []
for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    if not cls_dir.exists():
        continue
    for img_path in tqdm(list(cls_dir.iterdir()), desc=cls):
        if img_path.suffix not in EXTENSIONS:
            continue
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception as e:
            corrupted.append((img_path, str(e)))

if corrupted:
    print(f'\n{len(corrupted)} imagens corrompidas encontradas:')
    for path, err in corrupted:
        print(f'  {path}: {err}')
else:
    print('Nenhuma imagem corrompida encontrada.')

## 5. Organizar em splits train/val/test estratificados

In [ ]:
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
# TEST_RATIO  = 0.10  (restante)
SEED = 42
random.seed(SEED)

split_counts = {'train': {}, 'val': {}, 'test': {}}

for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    if not cls_dir.exists():
        continue

    imgs = [p for p in cls_dir.iterdir() if p.suffix in EXTENSIONS]
    # Remove corrompidas
    corrupted_paths = {p for p, _ in corrupted}
    imgs = [p for p in imgs if p not in corrupted_paths]
    random.shuffle(imgs)

    n = len(imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        'train': imgs[:n_train],
        'val':   imgs[n_train:n_train + n_val],
        'test':  imgs[n_train + n_val:],
    }

    for split, split_imgs in splits.items():
        dest_dir = DATA_DIR / split / cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img_path in split_imgs:
            shutil.copy2(img_path, dest_dir / img_path.name)
        split_counts[split][cls] = len(split_imgs)

print('Dataset organizado!\n')
for split in ['train', 'val', 'test']:
    total = sum(split_counts[split].values())
    print(f'{split} ({total} imagens):')
    for cls, count in split_counts[split].items():
        print(f'  {cls:<22}: {count}')

## 6. EDA — Distribuição de classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2D6A4F', '#40916C', '#52B788', '#74C69D', '#95D5B2', '#B7E4C7']

for ax, split in zip(axes, ['train', 'val', 'test']):
    classes = list(split_counts[split].keys())
    counts  = list(split_counts[split].values())
    bars = ax.bar(range(len(classes)), counts, color=colors)
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'{split} ({sum(counts)} imagens)')
    ax.set_ylabel('Quantidade')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribuição de Classes por Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 7. EDA — Amostras visuais

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, cls in zip(axes.flatten(), CLASSES):
    cls_dir = DATA_DIR / 'train' / cls
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if not imgs:
        ax.axis('off')
        continue
    sample = random.choice(imgs)
    img = Image.open(sample).convert('RGB').resize((256, 256))
    ax.imshow(img)
    ax.set_title(cls, fontsize=10)
    ax.axis('off')

plt.suptitle('Amostras de Cada Classe (train)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'samples.png', dpi=150)
plt.show()

## 8. EDA — Distribuição de tamanhos

In [ ]:
widths, heights = [], []
for cls in CLASSES:
    cls_dir = DATA_DIR / 'train' / cls
    for img_path in list(cls_dir.glob('*.jpg'))[:50]:  # amostra de 50 por classe
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception:
            pass

print(f'Largura  — média: {np.mean(widths):.0f} | min: {min(widths)} | max: {max(widths)}')
print(f'Altura   — média: {np.mean(heights):.0f} | min: {min(heights)} | max: {max(heights)}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color='#2D6A4F')
ax1.set_title('Distribuição de Largura (px)')
ax2.hist(heights, bins=30, color='#52B788')
ax2.set_title('Distribuição de Altura (px)')
plt.tight_layout()
plt.show()

print('\nDataset pronto para treinamento!')